# 13 · Checkpointers and threads (short-term memory)

**Goal:** by the end you can give a graph memory between `invoke` calls with a checkpointer, keep separate memories
with threads, inspect and fix the saved state, go back in time to an earlier checkpoint, and save memory to a file
that survives a restart.

## The idea in plain English

Every graph so far forgot everything when `invoke` returned. A **checkpointer** changes that. It saves a snapshot of
the state, called a **checkpoint**, after every step. The next time you invoke the graph, it starts from the last
saved state instead of from nothing.

Saves are grouped into **threads**. A thread is one separate line of memory, named by a **`thread_id`** you choose,
such as `"alice"`. Alice and Bob each get their own running total, even though they use the same graph.

Because every step is saved, you can also look back through the history, correct a saved value by hand, and even
restart from an old checkpoint to try something different. That last trick is called **time travel**.

**Analogy:** a video game with save slots. Each slot is a thread. The game autosaves after every step (a checkpoint).
You can load an old save and play on from there, and the original saves are still kept.

## Picture

Two threads, each with its own chain of checkpoints. Each `invoke` adds to the end of its own thread's chain.

```mermaid
flowchart LR
    subgraph alice["thread_id = alice"]
        a1["total 5"] --> a2["total 15"] --> a3["total 17.5"]
    end
    subgraph bob["thread_id = bob"]
        b1["total 1"]
    end
```

## Part 1: A running-total calculator

### Step 1: The state and the node

Each call passes an `amount`. The node adds it to the `total` so far. `state.get("total", 0.0)` returns `0.0` when
there's no total yet. `history` uses the `operator.add` reducer (lesson 06), so it grows by one line per call.

In [1]:
import operator
from typing import Annotated, TypedDict

from langgraph.graph import StateGraph, START, END


class TotalState(TypedDict):
    amount: float
    total: float
    history: Annotated[list[str], operator.add]


def add_amount(state: TotalState) -> dict:
    new_total = state.get("total", 0.0) + state["amount"]
    return {"total": new_total, "history": [f"+{state['amount']:g} -> {new_total:g}"]}

### Step 2: Without a checkpointer, nothing is remembered

Two calls, and the second one starts from zero again, exactly like every graph so far.

In [2]:
builder = StateGraph(TotalState)
builder.add_node("add_amount", add_amount)
builder.add_edge(START, "add_amount")
builder.add_edge("add_amount", END)

forgetful = builder.compile()
print(forgetful.invoke({"amount": 5}))
print(forgetful.invoke({"amount": 10}))

{'amount': 5, 'total': 5.0, 'history': ['+5 -> 5']}
{'amount': 10, 'total': 10.0, 'history': ['+10 -> 10']}


### Step 3: Compile with a checkpointer

**`InMemorySaver`** is the simplest checkpointer. It keeps checkpoints in the computer's memory (they're lost when
the kernel stops). You pass it to `compile(checkpointer=...)`. It's the same builder as before. Only the compile call
changes.

In [3]:
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)
print(graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	add_amount(add_amount)
	__end__([<p>__end__</p>]):::last
	__start__ --> add_amount;
	add_amount --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



### Step 4: Run on a thread

The thread goes in the **config**, the second argument of `invoke` (lesson 07), under `"configurable"`. Now the
second call on Alice's thread starts from her saved total.

In [4]:
alice = {"configurable": {"thread_id": "alice"}}

print(graph.invoke({"amount": 5}, alice))
print(graph.invoke({"amount": 10}, alice))

{'amount': 5, 'total': 5.0, 'history': ['+5 -> 5']}
{'amount': 10, 'total': 15.0, 'history': ['+5 -> 5', '+10 -> 15']}


Notice what happened to each key on the second call. `amount` came from the new input (`10`). `total` and `history`
were loaded from the last checkpoint and then updated.

### Step 5: Threads are separate

Bob's thread has never been used, so it starts from nothing. Alice's thread still has her total.

In [5]:
bob = {"configurable": {"thread_id": "bob"}}

print("bob:  ", graph.invoke({"amount": 1}, bob))
print("alice:", graph.invoke({"amount": 2.5}, alice))

bob:   {'amount': 1, 'total': 1.0, 'history': ['+1 -> 1']}
alice: {'amount': 2.5, 'total': 17.5, 'history': ['+5 -> 5', '+10 -> 15', '+2.5 -> 17.5']}


## Part 2: Looking at saved state

### Step 6: `get_state`, the latest checkpoint

`graph.get_state(config)` returns a **`StateSnapshot`**, the latest checkpoint of that thread. The useful parts:

- `values`: the state itself.
- `next`: which nodes would run next. It's empty, `()`, because the last run finished.
- `config`: the thread **and** this checkpoint's own ID, `checkpoint_id`.
- `metadata`: `step` (a step counter for the thread) and `source` (what created the checkpoint).

In [6]:
snapshot = graph.get_state(alice)

print("values:  ", snapshot.values)
print("next:    ", snapshot.next)
print("config:  ", snapshot.config)
print("metadata:", snapshot.metadata)

values:   {'amount': 2.5, 'total': 17.5, 'history': ['+5 -> 5', '+10 -> 15', '+2.5 -> 17.5']}
next:     ()
config:   {'configurable': {'thread_id': 'alice', 'checkpoint_ns': '', 'checkpoint_id': '1f1bfc80-9e83-694e-8007-2c4df980cc39'}}
metadata: {'source': 'loop', 'step': 7, 'parents': {}}


### Step 7: `get_state_history`, every checkpoint

`graph.get_state_history(config)` lists **every** checkpoint in the thread, newest first. We print it as a small
table, with only the last 6 characters of each long checkpoint ID.

Each `invoke` saved **three** checkpoints:

- `input`: the input has just arrived. `next` is `('__start__',)`.
- `loop`, with `next = ('add_amount',)`: the input is merged, and the node is about to run.
- `loop`, with `next = ()`: the node has run and the run is finished.

In [7]:
def show_history(graph, config):
    print(f"{'id':6}  {'step':>4}  {'source':6}  {'next':16}  total")
    for snap in graph.get_state_history(config):
        checkpoint_id = snap.config["configurable"]["checkpoint_id"][-6:]
        print(f"{checkpoint_id:6}  {snap.metadata['step']:>4}  {snap.metadata['source']:6}  "
              f"{str(snap.next):16}  {snap.values.get('total')}")


show_history(graph, alice)

id      step  source  next              total
80cc39     7  loop    ()                17.5
d6183c     6  loop    ('add_amount',)   15.0
9b2235     5  input   ('__start__',)    15.0
ff70e4     4  loop    ()                15.0
786fdf     3  loop    ('add_amount',)   5.0
c785c7     2  input   ('__start__',)    5.0
8888e1     1  loop    ()                5.0
32f738     0  loop    ('add_amount',)   None
c930bf    -1  input   ('__start__',)    None


Read it from the bottom up: three checkpoints for `+5`, three for `+10`, three for `+2.5`. The `total` column grows
5 → 15 → 17.5.

## Part 3: Fixing saved state by hand

### Step 8: `update_state`

Suppose Alice's total should really be `100`. `graph.update_state(config, values)` writes a new checkpoint as if a
node had returned `values`. That means **reducers still apply**. Here `total` is replaced, and the line we pass for
`history` is *appended*.

In [8]:
graph.update_state(alice, {"total": 100.0, "history": ["manual fix: total set to 100"]})

fixed = graph.get_state(alice)
print("values:  ", fixed.values)
print("metadata:", fixed.metadata)

values:   {'amount': 2.5, 'total': 100.0, 'history': ['+5 -> 5', '+10 -> 15', '+2.5 -> 17.5', 'manual fix: total set to 100']}
metadata: {'source': 'update', 'step': 8, 'parents': {}}


The new checkpoint's `source` is `update`. The next call carries on from the corrected total.

In [9]:
print(graph.invoke({"amount": 1}, alice))

{'amount': 1, 'total': 101.0, 'history': ['+5 -> 5', '+10 -> 15', '+2.5 -> 17.5', 'manual fix: total set to 100', '+1 -> 101']}


## Part 4: Time travel

### Step 9: Go back to an earlier checkpoint

We use a new thread, Carol, so the history is short. She adds 5, then 10.

In [10]:
carol = {"configurable": {"thread_id": "carol"}}
graph.invoke({"amount": 5}, carol)
graph.invoke({"amount": 10}, carol)
show_history(graph, carol)

id      step  source  next              total
b565d8     4  loop    ()                15.0
f7cd99     3  loop    ('add_amount',)   5.0
b4be3d     2  input   ('__start__',)    5.0
c8a3fa     1  loop    ()                5.0
f19761     0  loop    ('add_amount',)   None
3827b7    -1  input   ('__start__',)    None


Now: "what if Carol had added 1000 instead of 10?" We find the checkpoint where her **first** run had finished
(`total` is 5 and `next` is empty). Its `config` holds that checkpoint's `checkpoint_id`.

In [11]:
after_first_run = next(
    snap for snap in graph.get_state_history(carol)
    if snap.values.get("total") == 5.0 and snap.next == ()
)
print(after_first_run.config)

{'configurable': {'thread_id': 'carol', 'checkpoint_ns': '', 'checkpoint_id': '1f1bfc80-9efd-6f60-8001-a777f7c8a3fa'}}


### Step 10: Run again from there

Invoking with **that** config starts from that checkpoint instead of the latest one. The `+10` step is skipped, and
`+1000` happens in its place.

In [12]:
branch = graph.invoke({"amount": 1000}, after_first_run.config)
print(branch)

{'amount': 1000, 'total': 1005.0, 'history': ['+5 -> 5', '+1000 -> 1005']}


This created a **fork**: a new branch of history starting from the old checkpoint. The thread's latest state is now
the new branch. But nothing was deleted. The `+10 → 15` checkpoints are still in the history. The table below adds a
`parent` column, the checkpoint each one came from. Notice that two checkpoints share the same parent: that's where
the history splits.

In [13]:
print(f"{'id':6}  {'parent':6}  {'step':>4}  {'next':16}  total")
for snap in graph.get_state_history(carol):
    me = snap.config["configurable"]["checkpoint_id"][-6:]
    parent = snap.parent_config["configurable"]["checkpoint_id"][-6:] if snap.parent_config else "-"
    print(f"{me:6}  {parent:6}  {snap.metadata['step']:>4}  {str(snap.next):16}  {snap.values.get('total')}")

print()
print("carol now:", graph.get_state(carol).values)

id      parent  step  next              total
b2e34f  dcce20     4  ()                1005.0
dcce20  b2c0e4     3  ('add_amount',)   5.0
b2c0e4  c8a3fa     2  ('__start__',)    5.0
b565d8  f7cd99     4  ()                15.0
f7cd99  b4be3d     3  ('add_amount',)   5.0
b4be3d  c8a3fa     2  ('__start__',)    5.0
c8a3fa  f19761     1  ()                5.0
f19761  3827b7     0  ('add_amount',)   None
3827b7  -         -1  ('__start__',)    None

carol now: {'amount': 1000, 'total': 1005.0, 'history': ['+5 -> 5', '+1000 -> 1005']}


## Part 5: Memory that survives a restart

### Step 11: `SqliteSaver`

`InMemorySaver` loses everything when the kernel stops. **`SqliteSaver`** (from the `langgraph-checkpoint-sqlite`
package) saves checkpoints in a **SQLite** database, a single file on disk. We put the file in the project folder.
`.gitignore` already ignores `*.db` files, so it's never committed.

`sqlite3` is Python's built-in SQLite library. `check_same_thread=False` lets LangGraph use the connection from more
than one part of the program, which it needs to do.

To make this notebook give the same output every time, we delete any old file first.

In [14]:
import sqlite3
from pathlib import Path

from langgraph.checkpoint.sqlite import SqliteSaver

DB_PATH = Path.cwd().parent / "lesson13_running_total.db"
DB_PATH.unlink(missing_ok=True)  # start this demo with a fresh file

conn = sqlite3.connect(DB_PATH, check_same_thread=False)
saved_graph = builder.compile(checkpointer=SqliteSaver(conn))

print(saved_graph.invoke({"amount": 7}, alice))
print(saved_graph.invoke({"amount": 3}, alice))
conn.close()
print("saved to", DB_PATH.name, "-", DB_PATH.stat().st_size, "bytes")

{'amount': 7, 'total': 7.0, 'history': ['+7 -> 7']}
{'amount': 3, 'total': 10.0, 'history': ['+7 -> 7', '+3 -> 10']}


saved to lesson13_running_total.db - 28672 bytes


This file has its own Alice thread, separate from the in-memory one above: the same `thread_id`, but a different
checkpointer.

### After a restart

To see the memory survive for real: use **Kernel → Restart Kernel**, then run **only the cells below**, starting
here. They don't depend on anything above, so they work in a fresh kernel. (Run them without restarting and they work
the same way.)

The first cell opens the file again and reads Alice's state, without running the graph.

In [15]:
import operator
import sqlite3
from pathlib import Path
from typing import Annotated, TypedDict

from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import StateGraph, START, END


class TotalState(TypedDict):
    amount: float
    total: float
    history: Annotated[list[str], operator.add]


def add_amount(state: TotalState) -> dict:
    new_total = state.get("total", 0.0) + state["amount"]
    return {"total": new_total, "history": [f"+{state['amount']:g} -> {new_total:g}"]}

Now open the database file again and compile the graph with a `SqliteSaver`. `get_state` reads Alice's thread straight from the file, without running anything.

In [16]:
DB_PATH = Path.cwd().parent / "lesson13_running_total.db"
conn = sqlite3.connect(DB_PATH, check_same_thread=False)

builder = StateGraph(TotalState)
builder.add_node("add_amount", add_amount)
builder.add_edge(START, "add_amount")
builder.add_edge("add_amount", END)
reopened = builder.compile(checkpointer=SqliteSaver(conn))

alice = {"configurable": {"thread_id": "alice"}}
print("loaded from disk:", reopened.get_state(alice).values)

loaded from disk: {'amount': 3, 'total': 10.0, 'history': ['+7 -> 7', '+3 -> 10']}


The total of 10 came back from the file. The next call carries on from it. (Run this cell again and the total keeps
growing, because every call is saved to the file.)

In [17]:
print(reopened.invoke({"amount": 1}, alice))
conn.close()

{'amount': 1, 'total': 11.0, 'history': ['+7 -> 7', '+3 -> 10', '+1 -> 11']}


## What just happened

- **Step 2** ran without a checkpointer. Both calls started from zero: totals `5.0` and `10.0`.
- **Steps 3–4** compiled the same builder with `InMemorySaver`. On thread `alice`, `+5` then `+10` gave `15.0`.
- **Step 5** showed thread `bob` starting from nothing (`1.0`) while `alice` went on to `17.5`.
- **Step 6** read the latest checkpoint with `get_state`: `next` was `()`, and `config` held the `checkpoint_id`.
- **Step 7** listed every checkpoint with `get_state_history`: three per `invoke` (`input`, `loop`, `loop`).
- **Step 8** used `update_state` to set Alice's total to `100.0`. The history line was appended by the reducer, and
  the next `+1` gave `101.0`.
- **Steps 9–10** went back to Carol's checkpoint after `+5` and ran `+1000` from there, giving `1005.0`. The old
  `+10` branch is still in the history, and two checkpoints share one parent where the history splits.
- **Step 11** saved Alice's thread to a SQLite file. A new connection and a new graph loaded `total: 10.0` from disk
  and carried on to `11.0`.

## Common mistakes

1. **Forgetting the `thread_id`.** A graph with a checkpointer needs to know which thread to use:
   `ValueError: Checkpointer requires one or more of the following 'configurable' keys: thread_id, checkpoint_ns,
   checkpoint_id`.
   Fix: pass `{"configurable": {"thread_id": "..."}}` as the config.
2. **Reusing a `thread_id` by accident.** There's **no error**. If two tests both use `"test"`, the second one starts
   from whatever the first left behind, and its numbers look wrong.
   Fix: use a new `thread_id` for each separate conversation or test.
3. **Expecting `InMemorySaver` to survive a restart.** Again **no error**: after a kernel restart every thread is
   simply empty.
   Fix: use `SqliteSaver` (or another saver that writes to disk or a database) when memory must last.

## Try it

1. Start a new thread `"dave"` on the in-memory `graph`. Add 1, then 2, then 3. What is the total, and how many
   checkpoints does `get_state_history` return for Dave? Why that number?
2. Time travel on Alice's in-memory thread: find the checkpoint just **before** the manual fix (the last one whose
   `total` is `17.5` and `next` is `()`), and add `50` from there. What's the new total? Is the `manual fix` line in
   the new history?

Answers are in [`solutions/exercises_solutions.ipynb`](../solutions/exercises_solutions.ipynb).

## Key terms

New terms from this lesson, also in [GLOSSARY.md](../GLOSSARY.md):

- **Checkpointer**: saves the state after every step so later runs can continue from it. Passed to `compile`.
- **Checkpoint**: one saved snapshot of the state, with its own `checkpoint_id`.
- **Thread** / **`thread_id`**: one separate line of saved history, named in `{"configurable": {"thread_id": ...}}`.
- **`InMemorySaver`**: a checkpointer that keeps checkpoints in memory. Lost when the program stops.
- **`StateSnapshot`**: what `get_state` returns: `values`, `next`, `config`, `metadata` and more.
- **`get_state` / `get_state_history`**: read the latest checkpoint / every checkpoint of a thread.
- **`update_state`**: writes a new checkpoint by hand, as if a node returned those values (reducers apply).
- **Time travel**: running a graph again from an earlier checkpoint, by passing its `checkpoint_id` in the config.
- **Fork**: the new branch of history that time travel creates. The old branch is kept.
- **`SqliteSaver`**: a checkpointer that saves to a SQLite database file, so memory survives a restart.